# Supervised vs unsupervised autoencoders on CICIDS2018: re-run in Google Colab

Re-runs the experiments behind **Table 1** (supervised vs unsupervised autoencoder) and **Table 5** (known vs unseen attacks) and compares them with the reference run. Nothing here changes the reference results; new results go to a `rerun/` folder.

**Before you start**
1. In Google Drive create a folder `cicids2018-ae-share` (in *My Drive*).
2. Put in it: `sup_vs_unsup_code_package.zip`, and a sub-folder `processed` holding these 8 files from the project's `data/processed/`: `X_train.npy`, `X_val.npy`, `X_test.npy`, `meta_train.pkl`, `meta_val.pkl`, `meta_test.pkl`, `scaler.json`, `phase2_summary.json` (about 3.8 GB in total; Drive's free 15 GB is enough).
3. Use a normal **CPU** runtime (Runtime > Change runtime type > CPU). The code does not use a GPU.
4. Run the cells from top to bottom. Keep this browser tab open during the full run (about 1 hour or more; Colab may be slower than the reference laptop). If the session disconnects, the run is lost and must be started again.

The quick trial (step 7) only proves that everything works; its numbers are not valid results.

## 1. Connect Google Drive and check the files

In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")
SHARE = "/content/drive/MyDrive/cicids2018-ae-share"

# exact byte sizes of the files in the reference project (catches half-finished uploads)
EXPECTED = {
    "X_train.npy": 2142261632, "X_val.npy": 337985120, "X_test.npy": 623189504,
    "meta_train.pkl": 642637073, "meta_val.pkl": 101269873, "meta_test.pkl": 186810698,
    "scaler.json": 6134, "phase2_summary.json": 8863,
}
ok = os.path.exists(f"{SHARE}/sup_vs_unsup_code_package.zip")
print("code zip:", "found" if ok else "MISSING")
for name, size in EXPECTED.items():
    path = f"{SHARE}/processed/{name}"
    have = os.path.getsize(path) if os.path.exists(path) else None
    status = "ok" if have == size else ("MISSING" if have is None else f"WRONG SIZE ({have} bytes, expected {size})")
    ok &= have == size
    print(f"{name:22s} {status}")
assert ok, "Fix the files in Drive (see the list above) and run this cell again."

ModuleNotFoundError: No module named 'google'

## 2. Unpack the code and copy the data to the Colab disk (a few minutes)

In [ ]:
!rm -rf /content/cicids2018-ae
!unzip -q -o "{SHARE}/sup_vs_unsup_code_package.zip" -d /content
!mkdir -p /content/cicids2018-ae/data/processed
!cp -r "{SHARE}/processed/." /content/cicids2018-ae/data/processed/
%cd /content/cicids2018-ae
!ls -la data/processed

: 

## 3. Install the exact package versions used in the reference run (a few minutes)

A separate Python environment is used so the data files (saved with pandas 3.0.5) load correctly.
If the first line fails with "No matching distribution" (Colab's Python is too old or too new for these versions), see the note in the next cell's output.

In [ ]:
!python --version
!python -m venv /content/venv
!/content/venv/bin/pip install -q numpy==2.5.2 pandas==3.0.5 scikit-learn==1.9.0 matplotlib==3.11.1 psutil==7.2.2 river==0.26.1
!/content/venv/bin/pip install -q torch --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

## 4. Check that everything loads (expected: 2,163,852 test rows)

In [ ]:
!/content/venv/bin/python -c "import torch, pandas, numpy, sklearn; print('torch', torch.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__); m = pandas.read_pickle('data/processed/meta_test.pkl'); print('meta_test rows:', len(m)); assert len(m) == 2163852, 'unexpected number of test rows'; print('OK')"

## 5. Show the commands that will run (nothing is trained)

In [ ]:
!/content/venv/bin/python run_sup_vs_unsup.py --dry-run

## 6. Quick trial (a few minutes; numbers are NOT valid results)

In [ ]:
!/content/venv/bin/python run_sup_vs_unsup.py --quick --out rerun_quick

## 7. Full run (about 1 hour or more; keep this tab open)

In [ ]:
!/content/venv/bin/python run_sup_vs_unsup.py

## 8. Table 1 and Table 5 from this run

In [ ]:
from IPython.display import Markdown, Image, display

for title, path in (("Table 1 (this run)", "rerun/reports/phase5_tables.md"), ("Table 5 (this run)", "rerun/reports/phase6_tables.md")):
    display(Markdown(f"# {title}\n\n" + open(path, encoding="utf-8").read()))
for fig in ("phase5_roc_pr_comparison.png", "phase6_unseen_detection.png", "phase3_score_distributions.png"):
    display(Image(f"rerun/results/{fig}"))

## 9. Side-by-side with the reference run (expect small differences, not identical numbers)

In [ ]:
import json

ref5 = json.load(open("results/phase5_comparison.json"))["models"]
new5 = json.load(open("rerun/results/phase5_comparison.json"))["models"]
ref6 = json.load(open("results/phase6_results.json"))["models"]
new6 = json.load(open("rerun/results/phase6_results.json"))["models"]

print("TABLE 1 (test set, seen attack types)")
print(f"{'model':18s} {'ROC-AUC ref':>12s} {'ROC-AUC new':>12s} {'F1 ref':>8s} {'F1 new':>8s}")
for m in ("unsup_mixed", "normal_only", "supervised", "supervised_recon"):
    print(f"{m:18s} {ref5[m]['ranking']['roc_auc']:12.3f} {new5[m]['ranking']['roc_auc']:12.3f} "
          f"{ref5[m]['primary']['f1']:8.3f} {new5[m]['primary']['f1']:8.3f}")

print("\nTABLE 5 (withheld attack types: Bot, DDoS-HOIC, SSH-BruteForce, DoS-Hulk)")
print(f"{'model':18s} {'unseen recall ref':>18s} {'new':>8s} {'known recall ref':>17s} {'new':>8s}")
for m in ("unsup_mixed", "normal_only", "supervised"):
    print(f"{m:18s} {ref6[m]['unseen']['primary']['recall']:18.4f} {new6[m]['unseen']['primary']['recall']:8.4f} "
          f"{ref6[m]['known']['primary']['recall']:17.4f} {new6[m]['known']['primary']['recall']:8.4f}")

## 10. Save the results to Google Drive (so they survive when Colab closes)

In [ ]:
import shutil, time

dest = f"{SHARE}/rerun_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.copytree("rerun", dest, ignore=shutil.ignore_patterns("models"))
print("saved to", dest)